> 📄 English: [agent-file-conversion-tutorial.ipynb](agent-file-conversion-tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-file-conversion-with-hushvert--agent-file-conversion-tutorial)

# 为你的 Agent 赋予文件转换能力（本地、私有）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/agent-file-conversion-with-hushvert/agent-file-conversion-tutorial.ipynb)

## 概述

生产环境中的 Agent 会收到模型无法直接读取格式的文件：docx、pptx、xlsx、pdf，以及长长的图片、音频和压缩包尾部。对此有两种答案：

- **天真的答案**：把每个文件都上传到转换 API。这确实可行，但它把笔记本电脑几毫秒就能完成的工作变成了一次计费网络往返，而且它把用户文件发送到服务器去执行那些根本不需要服务器的转换。
- **生产级答案**：在设备能够处理时本地转换，只为真正必须在服务端运行的部分计费。本地转换零成本、不增加延迟预算，并且文件始终留在本机——这对健康、法律、金融和人力资源类文件来说不是可有可无，而是必须。

本教程将生产级答案构建为一个双车道（two-lane）方案，并以大多数转换内容避而不谈的问题收尾：与流行的提取栈相比，重构（reconstruction）到底为你带来了什么？

### 你将学到什么

1. 双车道路由规则，以及哪些格式属于哪条车道
2. 使用开源 WebAssembly 引擎进行本地、浏览器内转换
3. 作为 Claude Code、Cursor 和其他 MCP host 的 MCP 工具调用来进行转换
4. 从 Python 使用托管 API 的流程：提交、上传、轮询、下载
5. 如何解读转换保真度数字，而不被夸大宣传所迷惑

## 双车道架构

![The two-lane conversion rule](assets/architecture-diagram.svg)

路由规则就是整个设计的核心：**绝不为设备自己能完成的转换付费——无论是金钱还是隐私。**

- **本地车道**：图片（PNG、JPG、WebP、AVIF、JXL）、HEIC 解码、音频、压缩包、PDF 页面操作（合并、渲染、提取文本）、CSV/JSON/YAML/XLSX 数据文件，以及 DOCX 转 HTML 预览，全部以 WebAssembly 在浏览器中运行。零服务器成本、零上传。
- **服务器车道**：Office 文档转 PDF、PDF 转 Word、PDF/PPTX/XLSX 转 LLM 就绪的 markdown，以及大视频转码，确实无法在浏览器中运行。这些会进入计费 API，作为 MCP 工具调用或 REST 调用。

下面的一切都按照这个顺序走过两条车道，因为顺序本身就是重点：服务器车道是兜底方案，而不是默认方案。

## 步骤 1：环境配置

### 安装依赖

这个 notebook 本身只需要三个小包：用于 API 调用的 `requests`、用于生成真实测试文档的 `python-docx`，以及用于加载 key 的 `python-dotenv`。

In [ ]:
!pip install -q requests python-docx python-dotenv

### 配置 API 凭据

本地车道不需要账号，也不需要 key。对于服务器车道的部分：

1. 在 [hushvert.com/developers/keys](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-file-conversion-with-hushvert--agent-file-conversion-tutorial&click=hushvert-keys&target=https%3A%2F%2Fhushvert.com%2Fdevelopers%2Fkeys%3Futm_source%3Dagents-towards-production%26utm_medium%3Dgithub%26utm_campaign%3Dtutorial&text=hushvert%20developer%20keys) 创建一个免费账号：登录是一次性的邮箱验证码，无需密码，创建账号也是同样的流程。
2. 确认你的邮箱，然后在该页面创建一个 API key（key 形如 `hv_live_...`）。
3. 将其设置为环境变量 `HUSHVERT_API_KEY` 或放在本地 `.env` 文件中（这个 notebook 旁边就有一个 `.env.template`）。

新账号包含每月免费的服务器转换额度；本教程无需付费即可完成。请将 key 视为计费机密：仅用于服务端环境，绝不要放进浏览器 bundle。

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("HUSHVERT_API_KEY"):
    os.environ["HUSHVERT_API_KEY"] = getpass("Enter your hushvert API key (hv_live_...): ")

## 步骤 2：本地车道——绝不离开设备的转换

本地车道是 [`@hushvert/engine`](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-file-conversion-with-hushvert--agent-file-conversion-tutorial&click=hushvert-engine-npm&target=https%3A%2F%2Fwww.npmjs.com%2Fpackage%2F%40hushvert%2Fengine&text=%40hushvert%2Fengine)，一个 MIT 许可的 TypeScript 库，它在用户浏览器中以 WebAssembly 运行图片、HEIC、音频、压缩包、PDF 页面操作和数据文件转换。这部分方案本质上就是 JavaScript（这条车道就是用户的设备），因此在这里它以你将在 Web 应用中发布的代码形式出现，而 notebook 的其余部分则从 Python 演练服务器车道。

![Which lane owns which format](assets/format-map.svg)

图片零配置：

```ts
import { convertFile } from '@hushvert/engine'

// HEIC, PNG, JPG, WebP, AVIF, JXL. No configuration required.
const jpg = await convertFile(
  heicFile,
  { from: 'heic', to: 'jpg', module: 'images' },
  (pct) => console.log(`${pct}%`),
)
// `jpg` is a Blob. It never left the browser.
```

较重的编解码器（音频、视频、压缩包、PDF 渲染）从你的应用所提供的 WASM worker 运行，通过一次性的 `configureEngine` 调用指向这些资源来接线。

这条车道有两个特性值得在代码评审中强制执行：

- **可证伪的隐私。** 在转换期间打开网络标签页，没有任何请求携带文件；完全断网后转换依然能完成。一个你可以验证的隐私声明，胜过你必须相信的声明。
- **零边际成本。** 转换量落在用户的 CPU 上，而不是你的算力账单上。一个规模化转换截图或语音备忘录的 Agent 产品，绝不应该为此向服务器付费。

一个坦诚的提醒：引擎自身的代码是 MIT 许可，但两个可选的编解码器依赖是 copyleft 许可（ffmpeg 核心是 GPL，HEIC 解码是 LGPL）。这些模块是懒加载的，因此无法承担这些义务的项目，只需避开音频/视频和 HEIC 路径即可。

## 步骤 3：Agent 车道——转换作为 MCP 工具调用

对于你不手写代码的 Agent（Claude Code、Cursor、Cline、Zed、Claude Desktop），服务器车道以 MCP 服务器的形式提供，即 [`@hushvert/mcp`](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-file-conversion-with-hushvert--agent-file-conversion-tutorial&click=hushvert-mcp-npm&target=https%3A%2F%2Fwww.npmjs.com%2Fpackage%2F%40hushvert%2Fmcp&text=%40hushvert%2Fmcp)。整个集成就是一个配置块：

```json
{
  "mcpServers": {
    "hushvert": {
      "command": "npx",
      "args": ["-y", "@hushvert/mcp"],
      "env": { "HUSHVERT_API_KEY": "hv_live_your_key_here" }
    }
  }
}
```

然后让 Agent "convert quarterly.pptx to markdown"，它就会在一次工具调用中转换文件，将结果写到输入文件旁边。

![One tool call, end to end](assets/mcp-tool-call.svg)

服务器暴露四个工具：

| 工具 | 它的作用 |
| --- | --- |
| `convert_file` | 将本地文件转换为另一种格式，并把输出写到输入文件旁边 |
| `convert_poll` | 完成一个超时了第一次调用等待预算的长时间转换（大视频） |
| `list_formats` | 列出 API 支持的仅服务器转换对 |
| `check_usage` | 显示剩余的免费转换次数和积分余额 |

生产中有三个行为很重要：

- **它替你强制执行双车道规则。** 向它请求一个客户端可转换的转换对（比如 HEIC 转 JPG），它会拒绝并指向引擎，因此 Agent 循环永远不会在设备本来就能完成的工作上悄悄消耗积分。
- **它是可沙箱化的。** 设置 `HUSHVERT_ALLOWED_DIR` 后，服务器只在该目录下读写；设置 `HUSHVERT_MAX_JOBS_PER_SESSION` 作为防止失控 Agent 循环的护栏。
- **它是预算感知的。** 在批量任务之前让 Agent 调用 `check_usage`，而不是中途才发现余额为空。

## 步骤 4：从 Python 使用托管 API

对于你自己的流水线，同样的服务器车道就是一个普通的 REST API。本节完全可以运行。

### 发现格式面

formats 端点是公开的，返回每个服务器转换对，形如 `docx-to-md` 这样的 slug，并附带其大小上限和积分成本。获取一次并把它当作路由表来用；硬编码格式列表正是双车道规则腐化的方式。

In [ ]:
import requests

HUSHVERT_API = "https://hushvert.com/api"

pairs = requests.get(f"{HUSHVERT_API}/v1/formats", timeout=30).json()["pairs"]
server_pairs = {p["pair"] for p in pairs}

md_inputs = sorted(p["from"] for p in pairs if p["to"] == "md")
print(f"{len(pairs)} server pairs available")
print("Convertible to LLM-ready markdown:", ", ".join(md_inputs))

### 转换客户端

API 是一个三步流程：**提交（submit）**转换对和字节数（你会得到一个 `jobId` 和一个预签名的 `uploadUrl`），用普通 PUT **上传（upload）**字节（它们直接进入对象存储，而非经过 API 宿主），然后**轮询（poll）**直到 `done` 并下载结果。可选的 `Idempotency-Key` 头让重试变得安全：用相同的 key 重新提交会返回同一个任务，绝不会重复计费。

![The hosted API flow](assets/conversion-flow.svg)

In [ ]:
import pathlib
import time


def convert_file(path: str, to: str, idempotency_key: str | None = None) -> bytes:
    """Convert a local file via the hushvert API: submit, upload, poll, download."""
    src = pathlib.Path(path)
    data = src.read_bytes()
    pair = f"{src.suffix.lstrip('.').lower()}-to-{to}"

    headers = {"Authorization": f"Bearer {os.environ['HUSHVERT_API_KEY']}"}
    if idempotency_key:
        headers["Idempotency-Key"] = idempotency_key

    submit = requests.post(
        f"{HUSHVERT_API}/v1/conversions",
        json={"pair": pair, "bytes": len(data)},
        headers=headers,
        timeout=30,
    )
    submit.raise_for_status()
    job = submit.json()

    requests.put(job["uploadUrl"], data=data, timeout=300).raise_for_status()

    while True:
        poll = requests.get(
            f"{HUSHVERT_API}/v1/conversions/{job['jobId']}",
            headers=headers,
            timeout=30,
        )
        poll.raise_for_status()
        state = poll.json()
        if state["status"] == "done":
            result = requests.get(state["downloadUrl"], timeout=300)
            result.raise_for_status()
            return result.content
        if state["status"] == "failed":
            raise RuntimeError(f"Conversion failed: {state.get('error')}")
        time.sleep(2)

### 转换一份真实的 Office 文档

为了让教程自包含，我们自己生成传入文件：一份 docx 格式的小型供应商评审报告，包含一个标题、一个段落和一个真正的 Word 表格——正是 Agent 在工单中会拿到的那种文档。

In [ ]:
from docx import Document

doc = Document()
doc.add_heading("Vendor Security Review, Q3", level=1)
doc.add_paragraph(
    "Four vendors were reviewed this quarter. One review remains open pending "
    "an updated penetration-test report."
)

table = doc.add_table(rows=5, cols=3)
table.style = "Table Grid"
rows = [
    ("Vendor", "Risk score", "Status"),
    ("Acme Cloud", "Low", "Approved"),
    ("DataBridge", "Medium", "Approved with conditions"),
    ("Sky CRM", "High", "Rejected"),
    ("OptiLog", "Medium", "Open"),
]
for r, row in enumerate(rows):
    for c, text in enumerate(row):
        table.rows[r].cells[c].text = text

doc.save("vendor_review.docx")
print("Wrote vendor_review.docx")

### 运行转换并检查 Markdown

一次调用即可将 docx 转换为 markdown。值得细看的是表格：它以 GFM 管道表格的形式返回，因此读取此输出的模型仍然可以逐行回答"哪个供应商被拒绝，以及为什么 OptiLog 需要跟进"，而不是在一串被压平的单元格值中猜测。

In [ ]:
markdown = convert_file("vendor_review.docx", to="md").decode("utf-8")
print(markdown)

## 步骤 5：直说保真度问题

任何转换教程都欠你一个答案："这真的比我已在使用的提取栈更好吗？" 以下是坦诚的版本，来自对这套 API 的 PDF 转 markdown 链（重构：先 pdf2docx，再 pandoc 的 GFM writer）与 markitdown 0.1.6（流行的基于 pdfminer/pdfplumber 的提取器）的基准测试。语料：生成已知 ground-truth 单元格值的测试夹具（机械计数，而非目测），覆盖带边框、无边框、合并单元格以及跨页 60 行的表格、双栏布局，以及混合多元素文档。

| 夹具 | 重构链 | markitdown（基于 pdfminer） |
| --- | --- | --- |
| 带边框表格 | 4/4 行保留 | 4/4 行保留 |
| 无边框表格 | 7/7 | 7/7 |
| 合并单元格 | 6/6 | 6/6 |
| 跨页 60 行表格 | 60/60 | 60/60 |
| 双栏布局 | 阅读顺序正确（10/10 标记） | 阅读顺序错乱（10/10 标记） |

两个发现，而第一个正是大多数营销文案会隐藏的：

1. **在表格上，现代本地提取器已经赶上了。** 在每个表格夹具上，包括困难的那些，markitdown 都与重构链打平。如果你的语料是带表格的单栏文档，免费的本地提取器就够用了，任何告诉你相反结论的人都是在推销东西。
2. **剩余的差异是阅读顺序。** 在双栏夹具上，提取把两栏交错在了一起（10 个顺序标记每一个都错位了），而重构正确读取了两栏。现实中的报告、论文和合同充满了多栏布局，交错的文本会在不抛出任何错误的情况下，悄然毒害下游的一切（分块、embedding、回答）。

![The benchmark result illustrated](assets/reading-order.svg)

让这些数字保持诚实的注意事项：在小型合成但贴近现实的语料上单次运行的墙钟时间；这套 API 的 PPTX 和 XLSX markdown 车道本身就封装了 markitdown，因此那里的持平是结构性的；没有文本层的扫描 PDF 会快速失败，抛出清晰错误且不计费，因为该链读取文本层而不做 OCR。

路由的结论是：当你的输入包含真实布局时选择重构，不包含时可以随意本地提取。这是同一条双车道逻辑在更深一层的应用。

## 生产环境考量

- **计量与配额。** 服务器转换按次计费：先使用每月免费额度，然后使用预付费积分。`GET /v1/usage` 返回剩余额度和余额；批量任务应事先检查，而不是中途失败。
- **大小上限。** 每个转换对都带有上限（formats 端点以 `freeMaxBytes` 返回）；超大的上传会在产生任何费用前被 HTTP 413 拒绝。
- **留存。** 上传的输入在转换完成的那一刻即被删除；输出大约一小时后过期。下载 URL 是一次交接，而非存储：自行获取结果并持久化。
- **Key 卫生。** API key 是计费机密。将它保存在服务端环境中；如果浏览器应用需要重量级转换，在 API 前面放一个你自己的轻量代理，把 key 藏在代理后面。本地车道完全不需要 key。
- **幂等性。** 用稳定的 `Idempotency-Key` 将 `convert_file` 调用包裹在重试逻辑中，不稳定的网络就绝不可能对同一任务重复计费。
- **在评审中强制执行车道划分。** 一个为客户端可转换的转换对添加服务器转换的 pull request，既是成本 bug，也是隐私 bug。MCP 服务器会在运行时拒绝此类调用；你的评审清单则应在合并时拒绝它们。

## 总结

你为生产 Agent 构建了一个双车道文档接入方案：

- **本地车道**：一个 MIT 许可的 WebAssembly 引擎在用户设备上转换图片、HEIC、音频、压缩包、PDF 页面操作和数据文件，可验证地无需上传，零边际成本。
- **Agent 车道**：一个 MCP 配置块就能让任何 MCP host 拥有处理重量级格式的 `convert_file` 工具，客户端可转换的转换对在设计上被拒绝。
- **流水线车道**：一个四步 REST 流程（提交、上传、轮询、下载），带幂等重试和诚实的失败状态。
- **基于基准测试的路由规则**：对于单栏文档，提取就足够了；当布局真实存在时，重构才物有所值，因为阅读顺序正是提取会失效的地方。

更大的经验超越了文件转换本身：对于你赋予 Agent 的任何能力，问问哪部分可以在数据已经所在的地方运行，只为剩余部分计费。

接下来去哪里：面向开发者的 [hushvert](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-file-conversion-with-hushvert--agent-file-conversion-tutorial&click=hushvert-developers&target=https%3A%2F%2Fhushvert.com%2Ffor-developers%3Futm_source%3Dagents-towards-production%26utm_medium%3Dgithub%26utm_campaign%3Dtutorial&text=hushvert%20for%20developers) 页面记录了 API 面和定价，而[引擎源码](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-file-conversion-with-hushvert--agent-file-conversion-tutorial&click=hushvert-engine-github&target=https%3A%2F%2Fgithub.com%2Fhushvert%2Fengine&text=engine%20source)是开放的，如果你想审计"零上传"声明或添加模块的话。